In [ ]:
# general
import sys 
import os 
from pathlib import Path
# virtual cell
from gears import GEARS, PertData
import scanpy as sc
# data 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
# modelling 

# global settings
seed = 1733
np.random.seed(seed)
vcc_data_path = Path('vcc_data/adata_Training.h5ad')
gears_data_path = Path('gears_data')


In [ ]:
# laod prepared data
pert_data = PertData(gears_data_path)
pert_data.load(data_path=str(gears_data_path/"vcc_data"))
# creating masks for training and validation
pert_data.prepare_split(split='simulation', seed=1, )
# creating data loader for training
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)

Model

In [ ]:
gears_model = GEARS(
    pert_data,
    device='cuda',
    weight_bias_track=False,
    proj_name='vcc',
    exp_name='vcc'
)

gears_model.load_pretrained('test_model_32')


Odczyt metryk

In [ ]:
# Funkcja parsująca
def parse_log(data_log):
    models = []
    current_model = {'epochs': [], 'test_top20_de_mse': None, 'subgroup': {}}
    lines = data_log.split('\n')
    epoch = 0
    in_training = False
    in_testing = False
    in_subgroup = False
    
    for line in lines:
        if 'Start Training...' in line:
            if current_model['epochs']:  # Zapisz poprzedni model, jeśli istnieje
                models.append(current_model)
                current_model = {'epochs': [], 'test_top20_de_mse': None, 'subgroup': {}}
            in_training = True
            continue
        if 'Done!' in line and in_training:
            in_training = False
            continue
        if 'Start Testing...' in line:
            in_testing = True
            continue
        if 'Start doing subgroup analysis' in line:
            in_subgroup = True
            in_testing = False
            continue
        if 'Done!' in line and in_subgroup:
            in_subgroup = False
            continue  # Nie appenduj jeszcze, bo może być więcej
        
        if in_training:
            overall_match = re.search(r'Epoch (\d+): Train Overall MSE: (\d+\.\d+) Validation Overall MSE: (\d+\.\d+)\.', line)
            if overall_match:
                epoch = int(overall_match.group(1))
                train_mse = float(overall_match.group(2))
                val_mse = float(overall_match.group(3))
                current_model['epochs'].append({'epoch': epoch, 'train_overall_mse': train_mse, 'val_overall_mse': val_mse})
                continue
            
            de_match = re.search(r'Train Top 20 DE MSE: (\d+\.\d+) Validation Top 20 DE MSE: (\d+\.\d+)\.', line)
            if de_match and current_model['epochs']:
                train_de_mse = float(de_match.group(1))
                val_de_mse = float(de_match.group(2))
                current_model['epochs'][-1]['train_top20_de_mse'] = train_de_mse
                current_model['epochs'][-1]['val_top20_de_mse'] = val_de_mse
                continue
        
        if in_testing:
            test_de_match = re.search(r'Best performing model: Test Top 20 DE MSE: (\d+\.\d+)', line)
            if test_de_match:
                current_model['test_top20_de_mse'] = float(test_de_match.group(1))
                continue
        
        if in_subgroup:
            metric_match = re.search(r'(\w+): (nan|\d+\.\d+)', line)
            if metric_match:
                key = metric_match.group(1)
                value = metric_match.group(2)
                if value != 'nan':
                    current_model['subgroup'][key] = float(value)
    
    if current_model['epochs']:  # Dodaj ostatni model
        models.append(current_model)
    
    # Konwersja na struktury danych
    results = {}
    hidden_sizes = [32 + 16 * i for i in range(6)]  # 32,48,64,80,96,112
    for idx, model in enumerate(models):
        hs = hidden_sizes[idx] if idx < len(hidden_sizes) else f'unknown_{idx+1}'
        df_epochs = pd.DataFrame(model['epochs'])
        subgroup_dict = model['subgroup']
        if model['test_top20_de_mse'] is not None:
            subgroup_dict['best_test_top20_de_mse'] = model['test_top20_de_mse']
        results[hs] = {'epochs_df': df_epochs, 'test_metrics': subgroup_dict}
    
    return results

In [ ]:
data_log = open('trainoutput.txt', 'r').read()
information_df = parse_log(data_log)

In [ ]:
information_df[32]['epochs_df']

Tworzenie analiz

In [ ]:
# Tworzenie folderu
output_dir = Path('gears_comparison')
output_dir.mkdir(exist_ok=True)

# 1. Test Metrics DataFrame: wiersze = hidden sizes, kolumny = metryki
test_metrics_list = []
for hidden_size, data in information_df.items():
    metrics_row = data['test_metrics'].copy()
    metrics_row['hidden_size'] = hidden_size  # Dodaj jako kolumnę dla indeksu
    test_metrics_list.append(metrics_row)

test_df = pd.DataFrame(test_metrics_list)
test_df.set_index('hidden_size', inplace=True)
test_df.to_csv(output_dir / 'test_metrics_comparison.csv')
print("Zapisano test_metrics_comparison.csv")

# 2. Dla epochs_df: Osobne wykresy per model
for hidden_size, data in information_df.items():
    epochs_df = data['epochs_df']
    
    fig, axs = plt.subplots(2, 1, figsize=(10, 8))
    
    # Overall MSE
    axs[0].plot(epochs_df['epoch'], epochs_df['train_overall_mse'], label='Train Overall MSE', marker='o')
    axs[0].plot(epochs_df['epoch'], epochs_df['val_overall_mse'], label='Val Overall MSE', marker='s')
    axs[0].set_title(f'Model Hidden Size {hidden_size} - Overall MSE vs Epoch')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('MSE')
    axs[0].legend()
    axs[0].grid(True)
    
    # Top20 DE MSE
    axs[1].plot(epochs_df['epoch'], epochs_df['train_top20_de_mse'], label='Train Top20 DE MSE', marker='o')
    axs[1].plot(epochs_df['epoch'], epochs_df['val_top20_de_mse'], label='Val Top20 DE MSE', marker='s')
    axs[1].set_title(f'Model Hidden Size {hidden_size} - Top20 DE MSE vs Epoch')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('MSE')
    axs[1].legend()
    axs[1].grid(True)
    
    plt.tight_layout()
    plt.savefig(output_dir / f'epochs_plot_hidden_{hidden_size}.png')
    plt.close()
    print(f"Zapisano epochs_plot_hidden_{hidden_size}.png")

# 3. Jedna tabela łącząca train-val dla wszystkich modeli: pivot na modele
# Najpierw połącz wszystkie epochs_df z kolumną hidden_size
all_epochs_list = []
for hidden_size, data in information_df.items():
    epochs_copy = data['epochs_df'].copy()
    epochs_copy['hidden_size'] = hidden_size
    all_epochs_list.append(epochs_copy)

all_epochs_df = pd.concat(all_epochs_list, ignore_index=True)

# Pivot: wiersze = epoch, kolumny = hidden_size + metryka (np. hidden32_train_overall_mse)
# Aby uprościć, pivot dla każdej metryki osobno, ale zrób jeden szeroki DF
pivot_cols = ['train_overall_mse', 'val_overall_mse', 'train_top20_de_mse', 'val_top20_de_mse']
combined_df = pd.DataFrame({'epoch': all_epochs_df['epoch'].unique()})  # Wiersze epoki

for col in pivot_cols:
    pivot = all_epochs_df.pivot(index='epoch', columns='hidden_size', values=col)
    pivot.columns = [f'{col}_hidden_{col_name}' for col_name in pivot.columns]
    combined_df = pd.concat([combined_df, pivot], axis=1)

combined_df.set_index('epoch', inplace=True)
combined_df.to_csv(output_dir / 'epochs_comparison_table.csv')
print("Zapisano epochs_comparison_table.csv")

# Opcjonalnie: Zapisz jako Excel dla lepszej czytelności
with pd.ExcelWriter(output_dir / 'gears_comparison_summary.xlsx') as writer:
    test_df.to_excel(writer, sheet_name='Test Metrics')
    combined_df.to_excel(writer, sheet_name='Epochs Comparison')
print("Zapisano gears_comparison_summary.xlsx")

print(f"Wszystko zapisano w folderze: {output_dir}")

Później do notatek:
- na co tutaj trzeba zwrócić uwagę:
    - test_unseen_single ... mówi nam o informacji na wszystkich testowych
    - test_unseen_[...]_de - to sa geny które znacząco zmieniaja ekspresji
    - Uwagi:
        - Do tych dwóch modeli warto zwrócić że wartości są bardzo wysokie ale to najprawdopodobniej wynika że większość genów się mało zmieniła a model nie był zbyt drastyczny w swoich przewidywaniach.  
    - test_unseen_[...]_delta - to jest sama zmiana (czyli właściwe zachowanie modelu)
    - opposite_direction_[...] informacja o tym jak często popełnia błąd jeżeli chodzi o dopasowanie znaku 
    - test_unseen_single_frac_sigma_below_1_non_dropout informacja o niepewności - jak to czytać. 
    - tej ostatniej miary też nie wiem jak czytać


Do tych danych co widzę jakie są moje spostrzeżenia 


Propozycje
- nie widac overfittingu, mozna dac nizszy learning rate, mozna bardziej wybiorczo geny wziac, mozna wziac mniej genow ale w większości te które mają dużą ekspresje. Żeby ten model jeszcze raz uczyć. Ponieważ nie widać związku ze złożonością, w zależnosci do ilości próbek możemy sieograniczyć do 32/48 ponieważ znacząca skraca czas treningu a model nie wykorzystuje dodatkowych połączeń. 